<a href="https://colab.research.google.com/github/sagar-kumar16/AI-Waste-Sorting-System/blob/main/AI_Waste_Classification_with_Webcam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [ ]:
!pip install tensorflow opencv-python matplotlib

Import Libraries

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os, shutil, random
from pathlib import Path
import zipfile

Download Dataset

In [ ]:
!git clone https://github.com/garythung/trashnet.git

Cloning into 'trashnet'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 45 (delta 6), reused 1 (delta 0), pack-reused 33 (from 1)
Receiving objects: 100% (45/45), 40.64 MiB | 35.39 MiB/s, done.
Resolving deltas: 100% (12/12), done.


Extract & Prepare Dataset

In [ ]:
zip_path = Path("trashnet/data/dataset-resized.zip")
extract_dir = Path("trashnet/data/dataset-resized")

if not extract_dir.exists():
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

src = extract_dir / "dataset-resized"
dst = Path("dataset")

class_dirs = [d for d in os.listdir(src) if (src/d).is_dir() and d != '__MACOSX']

for split in ["train", "val"]:
    for cls in class_dirs:
        (dst/split/cls).mkdir(parents=True, exist_ok=True)

for cls in class_dirs:
    images = list((src/cls).glob("*.jpg"))
    random.shuffle(images)
    split_idx = int(0.8 * len(images))
    train_imgs, val_imgs = images[:split_idx], images[split_idx:]

    for img in train_imgs:
        shutil.copy(img, dst/"train"/cls)
    for img in val_imgs:
        shutil.copy(img, dst/"val"/cls)

print("Dataset ready !")

Dataset ready !


Load Dataset

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224,224)
BATCH = 32

train_gen = ImageDataGenerator(rescale=1./255)
val_gen = ImageDataGenerator(rescale=1./255)

train = train_gen.flow_from_directory(
    "dataset/train",
    target_size=IMG_SIZE,
    batch_size=BATCH
)

val = val_gen.flow_from_directory(
    "dataset/val",
    target_size=IMG_SIZE,
    batch_size=BATCH
)

print("Classes:", train.class_indices)

Found 2019 images belonging to 6 classes.
Found 508 images belonging to 6 classes.
Classes: {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}


Build Model (MobileNetV2)

In [ ]:
base = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base.trainable = False

model = tf.keras.Sequential([
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(train.num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,726 (9.24 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Train Model

In [ ]:
history = model.fit(
    train,
    validation_data=val,
    epochs=5
)

Epoch 1/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 101s 2s/step - accuracy: 0.7068 - loss: 0.7971 - val_accuracy: 0.7579 - val_loss: 0.6324
Epoch 2/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 92s 1s/step - accuracy: 0.8579 - loss: 0.3928 - val_accuracy: 0.7894 - val_loss: 0.5881
Epoch 3/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 92s 1s/step - accuracy: 0.9158 - loss: 0.2621 - val_accuracy: 0.7835 - val_loss: 0.5794
Epoch 4/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 106s 2s/step - accuracy: 0.9505 - loss: 0.1759 - val_accuracy: 0.7933 - val_loss: 0.6034
Epoch 5/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.9604 - loss: 0.1387 - val_accuracy: 0.7972 - val_loss: 0.6414


Save Model (IMPORTANT)

In [ ]:
model.save("waste_model.h5")
print("✅ Model saved successfully")

✅ Model saved successfully


Download Model

In [ ]:
from google.colab import files
files.download("waste_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Install + Import (Demo Only)

In [ ]:
!pip install tensorflow opencv-python matplotlib

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

Upload Model

In [ ]:
from google.colab import files
uploaded = files.upload()

Load Model

In [ ]:
model = tf.keras.models.load_model("waste_model.h5")
print("✅ Model loaded instantly")

Define Class Labels

In [ ]:
class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

Webcam Capture Function

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
import base64

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = '📸 Capture';
      div.appendChild(button);

      const video = document.createElement('video');
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      await new Promise((resolve) => button.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getTracks().forEach(track => track.stop());
      div.remove();

      return canvas.toDataURL('image/jpeg', quality);
    }
  ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = base64.b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

Prediction Function

In [ ]:
def predict_and_sort(img_path):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224,224))
    x = tf.keras.preprocessing.image.img_to_array(img)/255.0
    x = np.expand_dims(x, axis=0)

    pred = model.predict(x)
    class_idx = np.argmax(pred)
    label = class_names[class_idx]
    confidence = np.max(pred) * 100

    bin_map = {
        "plastic": "🟡 Bin 1",
        "paper": "🔵 Bin 2",
        "metal": "⚙️ Bin 3",
        "glass": "🟢 Bin 4",
        "cardboard": "📦 Bin 5",
        "trash": "⚫ Bin 6"
    }

    assigned_bin = bin_map.get(label, "Unknown")

    print("\n==============================")
    print(f"♻️ Waste Type: {label.upper()}")
    print(f"📊 Confidence: {confidence:.2f}%")
    print(f"➡️ Sorted Into: {assigned_bin}")
    print("==============================\n")

    plt.imshow(img)
    plt.title(f"{label} → {assigned_bin}")
    plt.axis("off")
    plt.show()

Run Live System

In [ ]:
while True:
    img_path = take_photo()
    predict_and_sort(img_path)

    cont = input("Continue? (y/n): ")
    if cont.lower() != 'y':
        break